In [ ]:
import sys
sys.path.insert(0, r"C:\Repo\EnterpriseCloud\astro_datautility")
from include.Python.TELTools import getSnowFlakeConn,getAzureSecret
from datetime import date, datetime, timedelta
from zoneinfo import ZoneInfo
import calendar
from typing import Dict, Any
import os
import json
import requests
from cryptography.fernet import Fernet


In [ ]:
def call_api(url,files=None,data=None,params=None,json_payload=None,headers=None,method='POST'):
    try:
        method=method.upper()
        
        if method == 'GET':
            resp = requests.get(url, params=params, headers=headers, timeout=120)
        elif method == 'POST':
            resp = requests.post(url, files=files, data=data, json=json_payload, headers=headers, timeout=120)
        else:
            raise ValueError(f"Unsupported HTTP method: {method}")
        
        resp.raise_for_status()
        op = resp.json()
        print(json.dumps(op, indent=2))
        return op
    except requests.HTTPError:
        print(f"HTTP {resp.status_code}: {resp.text}")
        raise

def save_refresh_token(path, token):
    with open(path, "w", encoding="utf-8") as f:
        f.write(token)
        
def load_refresh_token(path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            return f.read().strip()
    except FileNotFoundError:
        return None
    
def encrypt_token(new_refresh_token: str, secret_key: str) -> str:
    encryption_key = secret_key.encode()
    cipher = Fernet(encryption_key)
    encrypted_bytes = cipher.encrypt(new_refresh_token.encode())
    return encrypted_bytes.decode()

In [ ]:
#globals
TENANT_BASE_URL = "https://idcs-9913457ea7ca4a358bbd8c1eb1ae0daa.identity.oraclecloud.com:443"
CLIENT_ID = 'f565dab722fa4811a6e9c80dda150dda'
ENV = "PROD" #TEST or PROD
headers = {
    "Content-Type": "application/x-www-form-urlencoded;charset=utf-8"
}

In [ ]:
#call 1 : one time stuff (device user auth)

device_url = f"{TENANT_BASE_URL}/oauth2/v1/device"
SCOPE = {"PROD":"urn:opc:serviceInstanceID=814257143urn:opc:resource:consumer::all offline_access",
         "TEST":"urn:opc:serviceInstanceID=814809076urn:opc:resource:consumer::all offline_access"}



device_form = {
    "response_type": "device_code",
    "scope": SCOPE.get(ENV,"TEST"),
    "client_id": CLIENT_ID,
}


resp = call_api(device_url,data=device_form,headers=headers,method='POST')




{'device_code': '0bfc68b03f9c4e4ebf199e61bd213171',
 'user_code': 'JKCRFXSP',
 'verification_uri': 'https://idcs-9913457ea7ca4a358bbd8c1eb1ae0daa.identity.oraclecloud.com:443/ui/v1/device',
 'expires_in': 300}

In [19]:
#call 2 : get token


headers = {
    "Content-Type": "application/x-www-form-urlencoded;charset=utf-8"
}

token_url = f"{TENANT_BASE_URL}/oauth2/v1/token"
token_form = {
    "grant_type": "urn:ietf:params:oauth:grant-type:device_code",
    "device_code": resp['device_code'],
    "client_id": CLIENT_ID,
}

resp = call_api(token_url,data=token_form,headers=headers,method='POST')
refresh_token = resp.get('refresh_token')



"""User registerred succesfully in case of succesful generation of refresh_token."""

{
  "access_token": "eyJ4NXQjUzI1NiI6InRHcGFPXy1SN01teGhZb2Uwbkh0MjctSzVfcTVqZndQY0o0eWZVUXVBYXciLCJ4NXQiOiJlb2lvLVVHWFBNYldqUFYzNm4tOTlsMTFpbVUiLCJraWQiOiJTSUdOSU5HX0tFWSIsImFsZyI6IlJTMjU2In0.eyJjbGllbnRfb2NpZCI6Im9jaWQxLmRvbWFpbmFwcC5vYzEucGh4LmFtYWFhYWFhaWNncW95YWFwbHBubmZ2YWV0ajRoYzZwem95aGg2M3Z3Z3hzbWxobDY0c3RjNzVqcjVscSIsInVzZXJfdHoiOiJBbWVyaWNhL0NoaWNhZ28iLCJzdWIiOiJ2YXJ1bi54LnRpd2FyaUB1cy50ZWwuY29tIiwidXNlcl9sb2NhbGUiOiJlbiIsInNpZGxlIjo0ODAsInVzZXIudGVuYW50Lm5hbWUiOiJpZGNzLTk5MTM0NTdlYTdjYTRhMzU4YmJkOGMxZWIxYWUwZGFhIiwiaXNzIjoiaHR0cHM6Ly9pZGVudGl0eS5vcmFjbGVjbG91ZC5jb20vIiwiZG9tYWluX2hvbWUiOiJ1cy1waG9lbml4LTEiLCJjYV9vY2lkIjoib2NpZDEudGVuYW5jeS5vYzEuLmFhYWFhYWFhYXV0ZGNsYmd6MnVtNnRzeDIzcmNjbWNocXRzZTVpNHVvN210bHcydGR5bXoyb2lrbGxtYSIsInVzZXJfdGVuYW50bmFtZSI6ImlkY3MtOTkxMzQ1N2VhN2NhNGEzNThiYmQ4YzFlYjFhZTBkYWEiLCJjbGllbnRfaWQiOiJmNTY1ZGFiNzIyZmE0ODExYTZlOWM4MGRkYTE1MGRkYSIsInVzZXJfaXNBZG1pbiI6dHJ1ZSwiZG9tYWluX2lkIjoib2NpZDEuZG9tYWluLm9jMS4uYWFhYWFhYWFqajZrZTNxcWh0YnB2ZGF4emhwMmpqeWg

'User registerred succesfully in case of succesful generation of refresh_token.'

In [ ]:

#Update query to run on the latest RUN ID // i.e manually update refresh token
SECRET_VALUE = getAzureSecret("ORACLE-EPBCS-TOKEN-ENCRYTION-KEY").value.strip()
encrypted_refresh_token = encrypt_token(refresh_token, SECRET_VALUE)

update_qry = f"""
UPDATE LZ_PROD.static_objects.ORACLE_TB_LOAD_AUDIT_PROD
SET step_return_data = '{{"refreshtoken":"{encrypted_refresh_token}"}}'
WHERE run_id = (
  SELECT MAX(run_id)
  FROM LZ_PROD.static_objects.ORACLE_TB_LOAD_AUDIT_PROD where ENV = '{ENV}'
)
  AND step_name = 'fetch_refresh_token'
  AND ENV = '{ENV}';
""".strip()

print(update_qry)




UPDATE LZ_PROD.static_objects.ORACLE_TB_LOAD_AUDIT_PROD
SET step_return_data = '{"refreshtoken":"gAAAAABqZ0RS1O14vb8oleD34Zqeazn5lVTYCz1MkYnj9YGFgbZysj6rPizBgCb2l7tUfEikaGiqqbmdhBbvk7BPlf_NJ71JlNsq8a0ZkLKx2HqM50g79G1YJSQnTfUCrDdOzHb4SPm2OBIFct0Th3aD29o8BoXi2amltcXcU0lxDYflNsmRseYluTBEBSmJ6xtwxDu0rlc8XeUZxbzCSa4cBZdTVTNOPIPatZ8A4mP__eepnBG6A6ng3E-6P7Ldzn_NKnjhh_UXpMAQkInpHSJ3rTcvNtNFGl01zV4cYW-KlRCDHn14IJQ="}'
WHERE run_id = (
  SELECT MAX(run_id)
  FROM LZ_PROD.static_objects.ORACLE_TB_LOAD_AUDIT_PROD
)
  AND step_name = 'fetch_refresh_token'
  AND ENV = 'PROD';


In [21]:
print(update_qry)


UPDATE LZ_PROD.static_objects.ORACLE_TB_LOAD_AUDIT_PROD
SET step_return_data = '{"refreshtoken":"gAAAAABqZ0RS1O14vb8oleD34Zqeazn5lVTYCz1MkYnj9YGFgbZysj6rPizBgCb2l7tUfEikaGiqqbmdhBbvk7BPlf_NJ71JlNsq8a0ZkLKx2HqM50g79G1YJSQnTfUCrDdOzHb4SPm2OBIFct0Th3aD29o8BoXi2amltcXcU0lxDYflNsmRseYluTBEBSmJ6xtwxDu0rlc8XeUZxbzCSa4cBZdTVTNOPIPatZ8A4mP__eepnBG6A6ng3E-6P7Ldzn_NKnjhh_UXpMAQkInpHSJ3rTcvNtNFGl01zV4cYW-KlRCDHn14IJQ="}'
WHERE run_id = (
  SELECT MAX(run_id)
  FROM LZ_PROD.static_objects.ORACLE_TB_LOAD_AUDIT_PROD
)
  AND step_name = 'fetch_refresh_token'
  AND ENV = 'PROD';
